In [1]:
# packages
using Markdown
using InteractiveUtils
using NonlinearSolve
using StaticArrays
# files
using PKAssetPrices
import PKAssetPrices.Static: Parametrization, Model
using GLMakie


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


### Defining scenarios 

In [5]:
scen1a = Static.@scenario Static.AssetPKQ begin
    γ = 0.80
end

Parametrization
───────────────
Model:      16 vars, 21 params, 16 eqs, 6 curves, 3 sheets
Params:     21 set / 21 declared
u0 length:  16

Parameters
──────────
d0 = 5.0
b  = 0.5
i1 = 0.05
W0 = 2.0
i0 = 0.01
n  = 0.15
s0 = 0.5
α  = 0.1
c  = 0.8
h  = 0.8
Nᶠ = 6.0
d1 = 8.0
p1 = 1.0
s1 = 1.0
γ0 = 0.0
AQ = 6.0
k  = 0.3
a  = 0.8
m  = 0.15
γ  = 0.8
gₐ = 0.03


### Show results

In [6]:
sol1 = solve_model(Static.AssetPKQ)
sol2 = solve_model(Static.AssetPKQ)
sol_scen1a = solve_model(scen1a);

In [8]:
Static.build_table_from_solutions([sol1, sol2, sol_scen1a], ["Q", "PQ", "Scen1a"])

Variable,Scenario 1: Q,Scenario 2: PQ,Scenario 3: Scen1a
Y,6.5660179146289535,6.5660179146289535,6.566017914628954
ND,3.2830089573144767,3.2830089573144767,3.283008957314477
D,4.103761196643096,4.103761196643096,4.103761196643096
i,0.09741726123444605,0.09741726123444605,0.09741726123444606
r,0.11202985041961297,0.11202985041961297,0.11202985041961297
P,1.748345224688921,1.748345224688921,1.7483452246889213
dL,3.6689598213283947,3.6689598213283947,4.247886117349272
dM,3.6689598213283947,3.6689598213283947,4.247886117349272
dR,1.1006879463985184,1.1006879463985184,1.2743658352047815
W,1.9003752442270883,1.9003752442270883,1.9003752442270885


### Get Balance Sheets

In [9]:
display(html"<h3>Baseline</h3>")
display(sol1.sheets)
display(html"<h3>with s2=1</h3>")
display(sol2.sheets)
display(html"<h3>with s2=0.8</h3>")
display(sol_scen1a.sheets)

HTML{String}("<h3>Baseline</h3>")

Assets,Value,Liabilities,Value
Deposits,3.67,Loans,3.67
Total,3.67,Total,3.67
Assets,Value,Liabilities,Value
Loans,3.67,Deposits,3.67
Reserves,1.1,Central Bank Credit,1.1
Total,4.77,Total,4.77
Assets,Value,Liabilities,Value
Central Bank Credit,1.1,Reserves,1.1
Total,1.1,Total,1.1


HTML{String}("<h3>with s2=1</h3>")

Assets,Value,Liabilities,Value
Deposits,3.67,Loans,3.67
Total,3.67,Total,3.67
Assets,Value,Liabilities,Value
Loans,3.67,Deposits,3.67
Reserves,1.1,Central Bank Credit,1.1
Total,4.77,Total,4.77
Assets,Value,Liabilities,Value
Central Bank Credit,1.1,Reserves,1.1
Total,1.1,Total,1.1


HTML{String}("<h3>with s2=0.8</h3>")

Assets,Value,Liabilities,Value
Deposits,4.25,Loans,4.25
Total,4.25,Total,4.25
Assets,Value,Liabilities,Value
Loans,4.25,Deposits,4.25
Reserves,1.27,Central Bank Credit,1.27
Total,5.52,Total,5.52
Assets,Value,Liabilities,Value
Central Bank Credit,1.27,Reserves,1.27
Total,1.27,Total,1.27


### Show Curves

In [20]:
solution = solve_model(Static.AssetPKQ)
r_eq = solution.variables[:r]

y_eq = solution.variables[:Y]

# Build a range of r values around the equilibrium for IS
r_min = r_eq * 0.2
r_max = r_eq * 3.0
r_range = range(r_min, r_max; length = 200)

# Build a range of Y values around the equilibrium for IR
y_min = y_eq * 0.2
y_max = y_eq * 3.0
y_range = range(y_min, y_max; length = 200)

# IS curve: r → Y  (plot as x=r, y=IS(r))
is_values = Float64[]
for r in r_range
    vars = copy(solution.variables)
    vars[:r] = r
    curves = Static.eval_curve(solution.model, vars)
    push!(is_values, curves.IS)
end
# IR curve: Y → r  (plot as x=IR(Y), y=Y so both axes match)
ir_r_values = Float64[]
for y in y_range
    vars = copy(solution.variables)
    vars[:Y] = y
    curves = Static.eval_curve(solution.model, vars)
    push!(ir_r_values, curves.IR)
end
f = Figure()
a = Axis(f[1, 1], title = "IS curve", xlabel = "r", ylabel = "Y")
lines!(a, ir_r_values, y_range)
lines!(a, r_range, is_values)
f

In [23]:
    # Get the equilibrium values
    AP_eq = solution.variables[:AP]
    AS_eq = solution.variables[:AS]

    # Build a range of r values around the equilibrium for IS
    AP_min = AP_eq * 0.2
    AP_max = AP_eq * 3.0
    AP_range = range(AP_min, AP_max; length = 200)


    # IS curve: r → Y  (plot as x=r, y=IS(r))
    ad_values = Float64[]
    as_values = Float64[]
    for AP in AP_range
        vars = copy(solution.variables)
        vars[:AP] = AP
        curves = Static.eval_curve(solution.model, vars)
        push!(ad_values, curves.AMD)
        push!(as_values, curves.AMS)
    end
f = Figure()
a = Axis(f[1, 1], title = "IS curve", xlabel = "Q", ylabel = "AP")
lines!(a, ad_values, AP_range)
lines!(a, AP_range, as_values)
f